In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 8 - Week 7
# --------------------------------------------------
# Use all accumulated observations stored in the Week 7 .npy files.
# Fit an ARD Matern GP, derive search widths from fitted lengthscales,
# generate local + wider + global candidates, then calibrate EI and UCB.

In [2]:
X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 8

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (46, 8)
Y shape: (46,)

Current best observed input: [0.095473 0.144081 0.145356 0.087993 0.834791 0.549036 0.181794 0.551605]
Current best observed output: 9.991464829113


In [3]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(8) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
0.924**2 * Matern(length_scale=[1.21, 2, 0.97, 2, 2, 2, 1.32, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co

In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [1.21039108 2.         0.97015495 2.         2.         2.
 1.32272376 2.        ]
Normalised inverse-lengthscale sensitivity: [0.16158537 0.09779075 0.2015982  0.09779075 0.09779075 0.09779075
 0.14786269 0.09779075]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
Wider search widths: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]


In [6]:
rng = np.random.default_rng(42)

local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(70000, 8)
)

wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(30000, 8)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(15000, 8)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 115000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.008]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 115000


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [9]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
print("\nEI calibration:\n")

for xi in [0.0, 0.001, 0.005, 0.01, 0.02]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [0.         0.         0.04342305 0.17399414 0.76301123 1.
 0.         0.78902789] 
 mean = 9.79738 
 std = 0.278443 
 EI = 0.03998362 

xi=0.001 
 candidate = [0.         0.         0.04342305 0.17399414 0.76301123 1.
 0.         0.78902789] 
 mean = 9.79738 
 std = 0.278443 
 EI = 0.03974129 

xi=0.005 
 candidate = [0.         0.         0.04342305 0.17399414 0.76301123 1.
 0.         0.78902789] 
 mean = 9.79738 
 std = 0.278443 
 EI = 0.03878316 

xi=0.01 
 candidate = [0.         0.         0.04342305 0.17399414 0.76301123 1.
 0.         0.78902789] 
 mean = 9.79738 
 std = 0.278443 
 EI = 0.03761044 

xi=0.02 
 candidate = [0.         0.         0.04342305 0.17399414 0.76301123 1.
 0.         0.78902789] 
 mean = 9.79738 
 std = 0.278443 
 EI = 0.03534678 



In [11]:
print("\nUCB calibration:\n")

for beta in [0.1, 0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB calibration:

beta=0.1 
 candidate = [0.15050093 0.09394735 0.16214994 0.12907143 0.81623579 0.50963584
 0.19749667 0.57713075] 
 mean = 10.00123 
 std = 0.047096 
 UCB = 10.005939 

beta=0.25 
 candidate = [0.1841811  0.10790325 0.14774908 0.16798394 0.82407136 0.50642485
 0.17487477 0.64093167] 
 mean = 9.999301 
 std = 0.063423 
 UCB = 10.015157 

beta=0.5 
 candidate = [0.18836363 0.03914546 0.13800117 0.16481448 0.83641481 0.44341919
 0.15375969 0.61762418] 
 mean = 9.991141 
 std = 0.083429 
 UCB = 10.032855 

beta=1.0 
 candidate = [0.1557332  0.04642015 0.05041134 0.16080962 0.88375641 0.30807769
 0.16725297 0.48242944] 
 mean = 9.948578 
 std = 0.137673 
 UCB = 10.086251 

beta=1.5 
 candidate = [0.         0.         0.04342305 0.17399414 0.76301123 1.
 0.         0.78902789] 
 mean = 9.79738 
 std = 0.278443 
 UCB = 10.215044 



In [12]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.15050093 0.09394735 0.16214994 0.12907143 0.81623579 0.50963584
 0.19749667 0.57713075]
mean = 10.001229588662886
std = 0.047095805953302886


In [13]:
# --------------------------------------------------
# Final Function 8 Week 7 selection
# --------------------------------------------------
#
# EI consistently selected a highly uncertain boundary-heavy point
# with a substantially lower predicted mean, indicating that EI was
# dominated by exploration.
#
# The highest GP predicted mean was 10.001230, which is above the
# current best observed value of 9.991465.
#
# UCB with beta=0.1 selected exactly the same candidate as the
# highest predicted mean. Beta=0.25 selected a nearby alternative,
# while larger beta values progressively moved towards higher
# uncertainty and lower predicted mean.
#
# I therefore use low-exploration UCB with beta=0.1 for Week 7.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week7_candidate = candidates[final_idx]

print("Week 7 Function 8 candidate:")
print(week7_candidate)

print("\nPredicted mean:", mu[final_idx])
print("Predicted std:", sigma[final_idx])
print("UCB:", UCB[final_idx])

portal = "-".join(f"{x:.6f}" for x in week7_candidate)

print("\nPortal format:")
print(portal)

Week 7 Function 8 candidate:
[0.15050093 0.09394735 0.16214994 0.12907143 0.81623579 0.50963584
 0.19749667 0.57713075]

Predicted mean: 10.001229588662886
Predicted std: 0.047095805953302886
UCB: 10.005939169258216

Portal format:
0.150501-0.093947-0.162150-0.129071-0.816236-0.509636-0.197497-0.577131
